world - patients, resources, clinical rules

Stratergic planning (before acting)

what if simulation

calling tools

monitoring + replanning loop



In [ ]:
import os
import json
from typing import Dict, Any, List, Optional
from dotenv import load_dotenv
from pydantic import BaseModel, Field
from tenacity import retry, stop_after_attempt, wait_exponential, retry_if_exception_type

load_dotenv()

from openai import OpenAI
import os

from dotenv import load_dotenv
load_dotenv()

# Make sure OPENAI_API_KEY is set in your environment.
client = OpenAI(api_key=os.getenv("OPENAI_API_KEY"), base_url=os.getenv("OPENAI_API_BASE"))



MODEL = "gpt-4o-mini"

print("Using deployment:", MODEL)


Using deployment: gpt-4o-mini


In [ ]:
WorldModel = Dict[str, Any]

world: WorldModel = {
    "time": "09:00",
    "resources": {
        "icu_beds_available": 1,
        "general_beds_available": 3,
        "doctors_available": 2,
        "ventilators_available": 1
    },
    "patients": {
        "P1": {"age": 67, "symptoms": ["chest_pain", "breathlessness"], "spo2": 86, "bp": "90/60", "allergies": ["DrugA"], "labs": {}, "severity": None},
        "P2": {"age": 40, "symptoms": ["fever", "cough"], "spo2": 93, "bp": "120/80", "allergies": [], "labs": {}, "severity": None},
        "P3": {"age": 55, "symptoms": ["dizziness"], "spo2": 95, "bp": "85/55", "allergies": ["Penicillin"], "labs": {}, "severity": None},
    },
    "clinical_guidelines": {
        "icu_spo2_threshold": 88,
        "shock_bp_systolic_threshold": 90,
        "drug_interactions": {
            "DrugX": ["DrugA"],  # example: DrugX contraindicated if allergic to DrugA
        }
    },
    "history": []  # traceability of decisions and events
}

world


{'time': '09:00',
 'resources': {'icu_beds_available': 1,
  'general_beds_available': 3,
  'doctors_available': 2,
  'ventilators_available': 1},
 'patients': {'P1': {'age': 67,
   'symptoms': ['chest_pain', 'breathlessness'],
   'spo2': 86,
   'bp': '90/60',
   'allergies': ['DrugA'],
   'labs': {},
   'severity': None},
  'P2': {'age': 40,
   'symptoms': ['fever', 'cough'],
   'spo2': 93,
   'bp': '120/80',
   'allergies': [],
   'labs': {},
   'severity': None},
  'P3': {'age': 55,
   'symptoms': ['dizziness'],
   'spo2': 95,
   'bp': '85/55',
   'allergies': ['Penicillin'],
   'labs': {},
   'severity': None}},
 'clinical_guidelines': {'icu_spo2_threshold': 88,
  'shock_bp_systolic_threshold': 90,
  'drug_interactions': {'DrugX': ['DrugA']}},
 'history': []}

In [ ]:
class PatientAssessment(BaseModel):
    patient_id: str
    severity: str = Field(..., description="low/medium/high/critical")
    rationale: str

class PlanStep(BaseModel):
    step_id: int
    action: str
    target: str
    tool: Optional[str] = None
    expected_outcome: str

class DeliberativePlan(BaseModel):
    goal: str
    assumptions: List[str]
    assessments: List[PatientAssessment]
    constraints: List[str]
    plan_steps: List[PlanStep]

class WhatIfOption(BaseModel):
    option_name: str
    description: str
    pros: List[str]
    cons: List[str]
    risk_level: str = Field(..., description="low/medium/high")
    expected_outcome: str

class WhatIfAnalysis(BaseModel):
    options: List[WhatIfOption]
    recommended_option: str
    justification: str


In [ ]:
class LLMError(Exception):
    pass

@retry(
    stop=stop_after_attempt(3),
    wait=wait_exponential(multiplier=1, min=1, max=8),
    retry=retry_if_exception_type(LLMError)
)
def llm_json(system: str, user: str) -> Dict[str, Any]:
    """Call the LLM and parse a STRICT JSON response."""
    try:
        resp = client.chat.completions.create(
            model=MODEL,
            temperature=0.2,
            messages=[
                {"role": "system", "content": system},
                {"role": "user", "content": user},
            ],
        )
        text = resp.choices[0].message.content.strip()
        return json.loads(text)
    except Exception as e:
        # Raising LLMError triggers Tenacity retry
        raise LLMError(str(e))


In [ ]:
def tool_assign_bed(patient_id: str, bed_type: str, world: WorldModel) -> Dict[str, Any]:
    key = "icu_beds_available" if bed_type == "ICU" else "general_beds_available"
    if world["resources"][key] <= 0:
        return {"ok": False, "error": f"No {bed_type} beds available"}
    world["resources"][key] -= 1
    world["history"].append(f"Assigned {bed_type} bed to {patient_id}")
    return {"ok": True, "result": f"{patient_id} assigned to {bed_type}"}

def tool_order_lab(patient_id: str, lab_name: str, world: WorldModel) -> Dict[str, Any]:
    # Fake lab result generator (in reality: API call)
    fake_results = {
        "troponin": 0.9 if "chest_pain" in world["patients"][patient_id]["symptoms"] else 0.1,
        "cbc": {"wbc": 14000 if "fever" in world["patients"][patient_id]["symptoms"] else 8000},
        "lactate": 3.5 if world["patients"][patient_id]["bp"].startswith("85") else 1.2
    }
    world["patients"][patient_id]["labs"][lab_name] = fake_results.get(lab_name, "pending")
    world["history"].append(f"Ordered {lab_name} for {patient_id}")
    return {"ok": True, "result": {lab_name: world["patients"][patient_id]["labs"][lab_name]}}

def tool_notify_doctor(message: str, world: WorldModel) -> Dict[str, Any]:
    world["history"].append(f"Doctor notified: {message}")
    return {"ok": True, "result": "Doctor notified"}

TOOLS = {
    "assign_bed": tool_assign_bed,
    "order_lab": tool_order_lab,
    "notify_doctor": tool_notify_doctor
}


In [ ]:
def update_world_model_with_llm(world: WorldModel) -> WorldModel:
    system = """You are a clinical triage planner.
Return ONLY valid JSON. No markdown. No extra text."""

    user = f"""
World state:
{json.dumps(world, indent=2)}

Task:
1) Assess each patient severity: low/medium/high/critical.
2) Provide rationale.
Return JSON in this format:
{{
  "assessments": [
    {{"patient_id":"P1","severity":"critical","rationale":"..."}},
    ...
  ]
}}
"""

    data = llm_json(system, user)

    for a in data["assessments"]:
        pid = a["patient_id"]
        world["patients"][pid]["severity"] = a["severity"]

    world["history"].append("Updated severities from LLM triage")
    return world


In [ ]:
def make_deliberative_plan(world: WorldModel) -> DeliberativePlan:
    system = """You are a deliberative agent that plans before acting.
Return ONLY valid JSON. No markdown. No extra text."""

    user = f"""
World state:
{json.dumps(world, indent=2)}

Goal:
Maximize patient safety and outcomes given limited ICU beds and staff.

Constraints:
- ICU bed availability must not go negative.
- If Spo2 < {world['clinical_guidelines']['icu_spo2_threshold']} consider ICU priority.
- If systolic BP < {world['clinical_guidelines']['shock_bp_systolic_threshold']} treat as shock risk.
- Respect drug allergy interactions in world['clinical_guidelines']['drug_interactions'].

Return JSON following this schema:
{DeliberativePlan.model_json_schema()}
"""

    data = llm_json(system, user)
    return DeliberativePlan(**data)


In [ ]:
def what_if_analysis(world: WorldModel, plan: DeliberativePlan) -> WhatIfAnalysis:
    system = """You are a risk-aware planner doing what-if analysis.
Return ONLY valid JSON. No markdown. No extra text."""

    user = f"""
World state:
{json.dumps(world, indent=2)}

Proposed plan:
{plan.model_dump_json(indent=2)}

Task:
Propose 3 realistic alternative strategies and compare:
- outcomes
- risks
- pros/cons
Then recommend one option.

Return JSON with schema:
{WhatIfAnalysis.model_json_schema()}
"""

    data = llm_json(system, user)
    return WhatIfAnalysis(**data)


In [ ]:
def execute_plan(world: WorldModel, plan: DeliberativePlan) -> Dict[str, Any]:
    execution_log = []

    for step in plan.plan_steps:
        if step.tool and step.tool in TOOLS:
            # Minimal demo argument parsing
            if step.tool == "assign_bed":
                bed_type = "ICU" if "ICU" in step.action.upper() else "General"
                result = TOOLS[step.tool](step.target, bed_type, world)

            elif step.tool == "order_lab":
                lab = "troponin" if "troponin" in step.action.lower() else "cbc"
                result = TOOLS[step.tool](step.target, lab, world)

            elif step.tool == "notify_doctor":
                result = TOOLS[step.tool](step.action, world)

            else:
                result = {"ok": False, "error": "Unknown tool mapping"}

        else:
            # No tool → internal reasoning step
            result = {"ok": True, "result": f"Internal step: {step.action}"}

        execution_log.append({
            "step_id": step.step_id,
            "action": step.action,
            "tool": step.tool,
            "result": result
        })

        # Monitoring: stop and replan on resource failure
        if step.tool == "assign_bed" and not result["ok"]:
            return {"ok": False, "reason": "Execution failed (resource constraint)", "log": execution_log}

    return {"ok": True, "log": execution_log}


In [ ]:
def deliberative_agent_run(world: WorldModel, max_rounds: int = 3) -> WorldModel:
    for round_idx in range(1, max_rounds + 1):
        print(f"\n================ ROUND {round_idx} ================")

        # A) World model maintenance
        world = update_world_model_with_llm(world)
        print("Updated severities:", {k: v["severity"] for k, v in world["patients"].items()})

        # B) Plan
        plan = make_deliberative_plan(world)
        print("\nPlan goal:", plan.goal)

        # C) What-if analysis
        whatif = what_if_analysis(world, plan)
        print("\nRecommended option:", whatif.recommended_option)
        print("Justification:", whatif.justification)

        # D) Execute
        result = execute_plan(world, plan)
        print("\nExecution status:", result["ok"])
        for item in result["log"]:
            print(item)

        if result["ok"]:
            print("\nCompleted successfully.")
            break

        # E) Feedback → store failure & replan next round
        world["history"].append(f"Round {round_idx} failed: {result['reason']}")

    return world


In [ ]:
final_world = deliberative_agent_run(world, max_rounds=3)

print("\n========= FINAL WORLD HISTORY (last 20 events) =========")
for h in final_world["history"][-20:]:
    print("-", h)

print("\n========= FINAL RESOURCES =========")
print(final_world["resources"])

print("\n========= FINAL PATIENT SEVERITIES =========")
print({pid: p["severity"] for pid, p in final_world["patients"].items()})



================ ROUND 1 ================
Updated severities: {'P1': 'critical', 'P2': 'low', 'P3': 'medium'}

Plan goal: Maximize patient safety and outcomes given limited ICU beds and staff.

Recommended option: Discharge P2 for outpatient care
Justification: This option balances the need for critical care for P1 while managing resources effectively, allowing for potential new admissions without compromising the safety of P2.

Execution status: True
{'step_id': 1, 'action': 'Admit to ICU', 'tool': None, 'result': {'ok': True, 'result': 'Internal step: Admit to ICU'}}
{'step_id': 2, 'action': 'Monitor in general ward', 'tool': None, 'result': {'ok': True, 'result': 'Internal step: Monitor in general ward'}}
{'step_id': 3, 'action': 'Monitor in general ward', 'tool': None, 'result': {'ok': True, 'result': 'Internal step: Monitor in general ward'}}

✅ Completed successfully.

========= FINAL WORLD HISTORY (last 20 events) =========
- Updated severities from LLM triage

========= FINAL 